# VGGFace2 linear probe: final paper figure

This notebook retains only the analysis needed to reproduce the **VGGFace2 panel in Figure 2C** of the final paper:

1. evaluation loss across the 20 linear-probe epochs;
2. evaluation Macro-F1 across the 20 linear-probe epochs.

## Inputs

Four `history.json` files corresponding to the reported frozen-backbone linear probes:

- Baseline
- Fovea-Gaze
- Periph
- Periph-NF

If a history contains later exploratory fine-tuning rows, they are ignored:
this notebook explicitly selects probe-stage entries only. No fine-tuning results were evaluated or reported in the published paper; strictly linear probe only!

## Output

`vggface2_probe_loss_and_macroF1_portrait.pdf`

The historical display label
`Periph-non-NF` is renamed to the paper-facing label `Periph`;
the underlying run key is unchanged.

In [ ]:
import os
import glob
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt


# ---------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------
# Each value may be either:
#   1) a run directory containing history.json, or
#   2) a direct path to history.json.
#
# Edit these paths to match the released result layout.

RUNS = {
    "baseline":      Path("results/vggface2/baseline"),
    "fovea_gaze":    Path("results/vggface2/fovea-gaze"),
    "periph_nonTTM": Path("results/vggface2/periph"),
    "periph_ttm":    Path("results/vggface2/periph-nf"),
}

OUT_DIR = Path("outputs")
OUT_PDF = OUT_DIR / "vggface2_probe_loss_and_macroF1_portrait.pdf"

OUT_DIR.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------------------
# Final plotting parameters
# ---------------------------------------------------------------------

FIGSIZE_PORTRAIT = (6.6, 7.8)

FS_TITLE  = 16
FS_LABEL  = 14
FS_TICKS  = 12
FS_LEGEND = 12
LW_LINE   = 2.0

plt.rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "font.size": FS_TICKS,
    "axes.titlesize": FS_TITLE,
    "axes.labelsize": FS_LABEL,
    "xtick.labelsize": FS_TICKS,
    "ytick.labelsize": FS_TICKS,
    "legend.fontsize": FS_LEGEND,
    "figure.dpi": 200,
    "savefig.dpi": 300,
    "savefig.pad_inches": 0.02,
    "axes.xmargin": 0.0,
})

COLOR = {
    "baseline":      "tab:blue",
    "fovea_gaze":    "tab:orange",
    "periph_nonTTM": "tab:green",
    "periph_ttm":    "tab:red",
}

LABEL = {
    "baseline":      "Baseline",
    "fovea_gaze":    "Fovea-Gaze",
    "periph_nonTTM": "Periph",
    "periph_ttm":    "Periph-NF",
}

ORDER = [
    "baseline",
    "fovea_gaze",
    "periph_nonTTM",
    "periph_ttm",
]

# Hard clamp
XMIN, XMAX = 1.0, 20.0

In [ ]:
# ---------------------------------------------------------------------
# Load only the linear-probe stage
# ---------------------------------------------------------------------

PROBE_STAGE_NAMES = {
    "probe",
    "linear_probe",
    "linear-probe",
    "lp",
    "linearprobe",
}

CAND = {
    "x": [
        "stage_epoch",
        "epoch",
        "global_epoch",
        "step",
    ],

    "eval_macro_f1": [
        "eval_macro_f1",
        "macro_f1",
        "eval_macroF1",
        "macroF1",
        "eval_f1",
        "macro_f1_eval",
        "val_macro_f1",
        "val_macroF1",
    ],

    "eval_loss": [
        "eval_loss",
        "loss",
        "eval_ce",
        "ce",
        "val_loss",
        "valid_loss",
        "val_ce",
    ],
}


def find_history_json(run_dir):
    run_dir = Path(run_dir)

    if run_dir.is_file() and run_dir.name == "history.json":
        return run_dir

    if run_dir.is_dir():
        direct = run_dir / "history.json"
        if direct.is_file():
            return direct

        cands = list(run_dir.rglob("history.json"))
        if cands:
            return max(
                cands,
                key=lambda p: p.stat().st_mtime,
            )

    return None


def get_stage(entry):
    s = (
        entry.get("stage")
        or entry.get("stage_name")
        or entry.get("phase")
        or ""
    )
    return str(s).strip().lower()


def pick_key(entries, candidates):
    keys = set()

    for e in entries:
        if isinstance(e, dict):
            keys.update(e.keys())

    for k in candidates:
        if k in keys:
            return k

    # notebook's fuzzy contains-match fallback.
    lower = {
        k.lower(): k
        for k in keys
    }

    for want in candidates:
        wl = want.lower()

        for kl, orig in lower.items():
            if wl in kl:
                return orig

    return None


def extract_series(entries, y_key):
    if not entries:
        return None

    x_key = pick_key(
        entries,
        CAND["x"],
    )

    if x_key is None:
        x = np.arange(
            1,
            len(entries) + 1,
            dtype=float,
        )
    else:
        x = np.array(
            [
                e.get(x_key, np.nan)
                for e in entries
            ],
            dtype=float,
        )

    y = np.array(
        [
            e.get(y_key, np.nan)
            for e in entries
        ],
        dtype=float,
    )

    mask = (
        np.isfinite(x)
        & np.isfinite(y)
    )

    x = x[mask]
    y = y[mask]

    if len(x) < 2:
        return None

    order = np.argsort(x)

    return (
        x[order],
        y[order],
    )


def load_probe_entries(run_dir):
    hp = find_history_json(run_dir)

    if hp is None:
        raise FileNotFoundError(
            f"Could not locate history.json under: {run_dir}"
        )

    with hp.open("r") as f:
        hist = json.load(f)

    if not isinstance(hist, list) or not hist:
        raise RuntimeError(
            f"Expected a non-empty list in: {hp}"
        )

    probe = [
        e
        for e in hist
        if isinstance(e, dict)
        and get_stage(e) in PROBE_STAGE_NAMES
    ]

    if not probe:
        raise RuntimeError(
            f"No linear-probe entries found in: {hp}"
        )

    return probe, hp


def resolve_shared_metric_key(metric_name, probe_entries):
    candidates = CAND[metric_name]

    for k in candidates:
        ok = True

        for entries in probe_entries.values():
            keys = set().union(
                *[
                    e.keys()
                    for e in entries
                ]
            )

            if k not in keys:
                ok = False
                break

        if ok:
            return k

    return None


probe_entries = {}
hist_paths = {}

for name, run_dir in RUNS.items():
    probe, hp = load_probe_entries(run_dir)

    probe_entries[name] = probe
    hist_paths[name] = hp

    print(
        f"[OK] {name}: "
        f"probe points={len(probe)} "
        f"history.json={hp}"
    )


# runs contain 20 probe epochs.
bad_counts = {
    name: len(entries)
    for name, entries in probe_entries.items()
    if len(entries) != 20
}

if bad_counts:
    raise RuntimeError(
        "Expected 20 linear-probe epochs for each final VGGFace2 run. "
        f"Found: {bad_counts}. Check that RUNS points to the final paper histories."
    )


shared_keys = {
    "eval_macro_f1": resolve_shared_metric_key(
        "eval_macro_f1",
        probe_entries,
    ),
    "eval_loss": resolve_shared_metric_key(
        "eval_loss",
        probe_entries,
    ),
}

print(
    "\n[INFO] Shared metric keys across all runs:",
    shared_keys,
)

In [ ]:
# ---------------------------------------------------------------------
# Figure 2C: VGGFace2 frozen-backbone linear probe
# ---------------------------------------------------------------------

def plot_metric_on_ax(
    ax,
    metric_name,
    ylabel,
    title,
    want_legend=False,
):
    any_curve = False

    for name in ORDER:
        entries = probe_entries[name]

        y_key = (
            shared_keys.get(metric_name)
            or pick_key(
                entries,
                CAND[metric_name],
            )
        )

        if y_key is None:
            raise RuntimeError(
                f"{name}: cannot find metric '{metric_name}'"
            )

        series = extract_series(
            entries,
            y_key,
        )

        if series is None:
            raise RuntimeError(
                f"{name}: insufficient data for key '{y_key}'"
            )

        x, y = series

        ax.plot(
            x,
            y,
            color=COLOR[name],
            linewidth=LW_LINE,
            alpha=0.90,
            label=LABEL[name],
        )

        any_curve = True

    ax.set_ylabel(
        ylabel
    )

    ax.set_title(
        title
    )

    ax.grid(
        True,
        alpha=0.30,
    )

    # hard clamp: exactly epochs 1 through 20.
    ax.set_xlim(
        XMIN,
        XMAX,
    )

    ax.margins(
        x=0
    )

    if want_legend and any_curve:
        ax.legend(
            loc="upper right",
            frameon=True,
            framealpha=0.9,
            borderpad=0.3,
        )


fig, axes = plt.subplots(
    2,
    1,
    figsize=FIGSIZE_PORTRAIT,
    sharex=True,
)


plot_metric_on_ax(
    axes[0],
    metric_name="eval_loss",
    ylabel="Eval loss",
    title="VGGFace2 linear probe: Eval loss",
    want_legend=True,
)


plot_metric_on_ax(
    axes[1],
    metric_name="eval_macro_f1",
    ylabel="Eval macro-F1",
    title="VGGFace2 linear probe: Eval macro-F1",
    want_legend=False,
)


axes[1].set_xlabel(
    "Probe epoch"
)


fig.tight_layout()

fig.savefig(
    OUT_PDF,
    format="pdf",
)

plt.show()

print(
    f"[OK] wrote: {OUT_PDF}"
)